In [1]:
# Install into the active notebook kernel
%pip install -q cassio datasets pyasyncore langchain langchain-community langchain-classic langchain-openai openai tiktoken python-dotenv

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
# LangChain components to use
from langchain_community.vectorstores import Cassandra
from langchain_classic.indexes.vectorstore import VectorStoreIndexWrapper
from langchain_openai import OpenAI, OpenAIEmbeddings

# Support for dataset retrieval with Hugging Face
from datasets import load_dataset

# With CassIO, the engine powering the Astra DB integration in LangChain,
# you will also initialize the DB connection:
import cassio

In [4]:
from pypdf import PdfReader

In [5]:
# Store Astra credentials outside the notebook.
# Required: ASTRA_DB_APPLICATION_TOKEN and either ASTRA_DB_ID or ASTRA_DB_SECURE_BUNDLE_PATH.
# Optional: ASTRA_DB_KEYSPACE

In [34]:
import os
from pathlib import Path

try:
    from dotenv import load_dotenv
except ImportError:
    load_dotenv = None

if load_dotenv is not None:
    for candidate in (
        Path('.env'),
        Path('Langchain/.env'),
        Path('vectordatabase_langchain/.env'),
        Path('../.env'),
        Path('../vectordatabase_langchain/.env'),
    ):
        if candidate.exists():
            load_dotenv(candidate, override=False)

ASTRA_DB_APPLICATION_TOKEN = 'AstraCS:eiNPHxDanJMKnUCFXHpjCFyc:73c4dac0f7d1bd599b54fa01e28e17e9abd64b8dfa134e11a16e4798d446f74f'
ASTRA_DB_ID = '7334a689-2bd0-4640-b910-4e0b9a76707b'
ASTRA_DB_KEYSPACE = 'default_keyspace'
ASTRA_DB_SECURE_BUNDLE_PATH = os.getenv('ASTRA_DB_SECURE_BUNDLE_PATH', '').strip()

if not ASTRA_DB_APPLICATION_TOKEN.startswith('AstraCS:'):
    raise ValueError('Set ASTRA_DB_APPLICATION_TOKEN in your environment or .env file.')

if ASTRA_DB_SECURE_BUNDLE_PATH and not Path(ASTRA_DB_SECURE_BUNDLE_PATH).expanduser().is_file():
    raise FileNotFoundError(f'Secure bundle not found: {ASTRA_DB_SECURE_BUNDLE_PATH}')

In [35]:
pdf_reader = PdfReader("../vectordatabase_langchain/budget_speech (1).pdf")

In [36]:
from typing_extensions import Concatenate
raw_text=''
for i, page in enumerate(pdf_reader.pages):
    text=page.extract_text()
    raw_text+=text

In [37]:
raw_text[:1000]

'GOVERNMENT OF INDIA\nBUDGET 2023-2024\nSPEECH\nOF\nNIRMALA SITHARAMAN\nMINISTER OF FINANCE\nFebruary 1,  2023CONTENTS \nPART-A \n Page No. \n\uf0b7 Introduction 1 \n\uf0b7 Achievements since 2014: Leaving no one behind 2 \n\uf0b7 Vision for Amrit Kaal – an empowered and inclusive economy 3 \n\uf0b7 Priorities of this Budget 5 \ni. Inclusive Development  \nii. Reaching the Last Mile \niii. Infrastructure and Investment \niv. Unleashing the Potential \nv. Green Growth \nvi. Youth Power  \nvii. Financial Sector \n \n \n \n \n \n \n \n \n \n\uf0b7 Fiscal Management \n24 \nPART B \n  \nIndirect Taxes 27 \n\uf0b7 Green Mobility  \n\uf0b7 Electronics   \n\uf0b7 Electrical   \n\uf0b7 Chemicals and Petrochemicals   \n\uf0b7 Marine products  \n\uf0b7 Lab Grown Diamonds  \n\uf0b7 Precious Metals  \n\uf0b7 Metals  \n\uf0b7 Compounded Rubber  \n\uf0b7 Cigarettes  \n  \nDirect Taxes 30 \n\uf0b7 MSMEs and Professionals   \n\uf0b7 Cooperation  \n\uf0b7 Start-Ups  \n\uf0b7 Appeals  \n\uf0b7 Better tar

In [27]:
OpenAI_API_KEY=""

In [38]:
cassio_kwargs = {
    'token': ASTRA_DB_APPLICATION_TOKEN,
    'keyspace': ASTRA_DB_KEYSPACE,
}

if ASTRA_DB_SECURE_BUNDLE_PATH:
    cassio_kwargs['secure_connect_bundle'] = str(Path(ASTRA_DB_SECURE_BUNDLE_PATH).expanduser())
else:
    cassio_kwargs['database_id'] = ASTRA_DB_ID

cassio.init(**cassio_kwargs)
print(f'CassIO initialized successfully for {ASTRA_DB_ID} / {ASTRA_DB_KEYSPACE}.')

CassIO initialized successfully for 7334a689-2bd0-4640-b910-4e0b9a76707b / default_keyspace.


In [ ]:
llm=OpenAI(openai_api_key=OpenAI_API_KEY, temperature=0.5)
embedding=OpenAIEmbeddings(openai_api_key=OpenAI_API_KEY)

vectorstore 